# Restormer teacher — Modal training

Same-resolution **256×256** restoration teacher (not NPU submission). Trains on Modal with `lab3-data` and `lab3-runs` volumes. The canonical implementation lives in `restormer_teacher/model.py`. See `docs/restormer_teacher_notes.md` for artifacts and distillation handoff.

## Setup

- Install and authenticate the [Modal](https://modal.com) CLI.
- Ensure `Data/` under this repo matches the Lab 3 layout (`HR_train*`, `LR_train*`, `HR_val`, `LR_val`).
- Optional env vars: `LAB3_MODAL_GPU`, `LAB3_NOTEBOOK_MODAL_DATA_VOLUME`, `LAB3_NOTEBOOK_MODAL_RUNS_VOLUME`, `MODAL_RUN_DAY`.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import yaml

NOTEBOOK_DIR = Path.cwd().resolve()
LAB3_ROOT = NOTEBOOK_DIR
for _cand in (NOTEBOOK_DIR, NOTEBOOK_DIR.parent, NOTEBOOK_DIR.parent.parent):
    if (_cand / "Data").is_dir():
        LAB3_ROOT = _cand
        break
PKG_ROOT = LAB3_ROOT / "Teacher-Student Reformer"
assert (LAB3_ROOT / "Data").is_dir(), f"Could not find Data/ under {LAB3_ROOT}"
assert PKG_ROOT.is_dir(), f"Expected {PKG_ROOT}"

sys.path.insert(0, str(PKG_ROOT))

from tools.restormer_teacher_modal_app import (
    DEFAULT_DATA_VOLUME_NAME,
    DEFAULT_GPU,
    DEFAULT_POLL_INTERVAL_MINUTES,
    DEFAULT_RUNS_VOLUME_NAME,
    DEFAULT_TIMEOUT_MINUTES,
    execute_restormer_teacher_modal,
)

CONFIG_PATH = PKG_ROOT / "configs" / "restormer_teacher.yaml"
RUN_NAME = os.environ.get("RESTORMER_RUN_NAME", "restormer_teacher_nb")
STARTED_DAY = os.environ.get("MODAL_RUN_DAY", "") or time.strftime("%Y-%m-%d", time.gmtime())
GPU = os.environ.get("LAB3_MODAL_GPU", DEFAULT_GPU)
DATA_VOL = os.environ.get("LAB3_NOTEBOOK_MODAL_DATA_VOLUME", DEFAULT_DATA_VOLUME_NAME)
RUNS_VOL = os.environ.get("LAB3_NOTEBOOK_MODAL_RUNS_VOLUME", DEFAULT_RUNS_VOLUME_NAME)
SMOKE = os.environ.get("RESTORMER_SMOKE", "").lower() in ("1", "true", "yes")

print("LAB3_ROOT", LAB3_ROOT)
print("CONFIG_PATH", CONFIG_PATH)
print("RUN_NAME", RUN_NAME, "DAY", STARTED_DAY, "GPU", GPU)

In [ ]:
raw_cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
raw_cfg.setdefault("training", {})["config_path"] = str(CONFIG_PATH)
print(json.dumps(raw_cfg.get("profiles", {}), indent=2)[:1200])

## Launch on Modal

Uncomment to run. This syncs `Data/` to the data volume (if needed), runs the remote trainer, then syncs `runs/<STARTED_DAY>/<RUN_NAME>/` locally. Logs and `history.json` / `metrics.jsonl` live in that folder.

In [ ]:
# summary = execute_restormer_teacher_modal(
#     raw_cfg,
#     local_lab3_root=LAB3_ROOT,
#     local_data_root=LAB3_ROOT / "Data",
#     run_name=RUN_NAME,
#     started_day=STARTED_DAY,
#     smoke_test=SMOKE,
#     sync_data=False,
#     force_data_sync=False,
#     gpu=GPU,
#     timeout_minutes=DEFAULT_TIMEOUT_MINUTES if not SMOKE else 90,
#     poll_interval_minutes=DEFAULT_POLL_INTERVAL_MINUTES,
#     data_volume_name=DATA_VOL,
#     runs_volume_name=RUNS_VOL,
# )
# print(json.dumps(summary, indent=2))

## After the run

- Open `runs/<STARTED_DAY>/<RUN_NAME>/history.json` for epoch curves.
- Tail `metrics.jsonl` for structured lines.
- Generate teacher targets:  
  `python "Teacher-Student Reformer/scripts/generate_teacher_targets.py" --checkpoint runs/.../checkpoints/best_ema.pth --data-root Data --output-dir runs/.../teacher_targets`